# 📘 Day 18: Classification Deep Dive (Portfolio-Level)
Complete theory + intuition + end-to-end hands-on pipeline


# 🔗 End-to-End Flow (Very Important)

Raw Data → Feature Engineering → Encoding → Model → Sigmoid/Softmax → Probability → Threshold → Final Prediction

This notebook connects ALL steps in one flow.



# 1. Feature Engineering (Advanced)

## Key Idea
Better features = better model

## Advanced Concepts
- Scaling improves convergence
- Log transform handles skew
- Feature interaction improves performance

## Real-world (Fraud Detection)
- Transaction amount
- Device type (categorical)
- Location


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Age': [22, 45, 25, 35, 52, 23],
    'Salary': [20000, 80000, 25000, 50000, 90000, 27000],
    'City': ['Kolkata', 'Delhi', 'Mumbai', 'Delhi', 'Mumbai', 'Kolkata'],
    'Fraud': [0,1,0,1,1,0]
})

df

💡 Simple Definition

Feature engineering is the process of converting raw data into meaningful inputs (features) that help a machine learning model learn better.

🔍 In Plain English

It’s how you prepare and improve data so the model can understand patterns easily.

🧱 Why It’s Important
Models don’t understand raw data directly
Good features → better predictions
Poor features → even the best model fails

👉 In many real projects:

Feature engineering matters more than the algorithm

🔧 Common Types of Feature Engineering

1️⃣ Numerical Transformation
Scaling (Standardization)
Normalization
Log transformation (for skewed data)

2️⃣ Categorical Encoding
One-hot encoding
Label encoding

3️⃣ Feature Creation
Age → Age group
Date → Day, Month, Weekend

4️⃣ Feature Selection
Remove irrelevant or redundant features
Helps reduce overfitting

🔥 Real-World Example (Fraud Detection)

Raw:

Amount = 50000
Time = 2 AM

Engineered:

High_amount_flag = 1
Night_transaction = 1

👉 Now model can easily detect suspicious patterns.

In [4]:
import pandas as pd

df = pd.DataFrame({
    'Amount': [500, 20000, 1500, 80000, 300],
    'Time': ['10:30', '02:15', '18:45', '01:10', '23:50'],
    'City': ['Kolkata', 'Delhi', 'Mumbai', 'Delhi', 'Kolkata'],
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Mobile', 'Desktop'],
    'Past_Transactions': [50, 2, 30, 1, 100]
})

df.head()

,Amount,Time,City,Device,Past_Transactions
0,500,10:30,Kolkata,Mobile,50
1,20000,02:15,Delhi,Desktop,2
2,1500,18:45,Mumbai,Mobile,30
3,80000,01:10,Delhi,Mobile,1
4,300,23:50,Kolkata,Desktop,100


In [7]:
#Extract Hour from Time
df['Hour'] = df['Time'].apply(lambda x: int(x.split(':')[0]))
df[['Time', 'Hour']]

,Time,Hour
0,10:30,10
1,02:15,2
2,18:45,18
3,01:10,1
4,23:50,23


In [8]:
# Create Night Transaction Feature

df['Is_Night'] = df['Hour'].apply(lambda x: 1 if x < 6 else 0)
df[['Hour', 'Is_Night']]

# 👉 Fraud often happens at night

,Hour,Is_Night
0,10,0
1,2,1
2,18,0
3,1,1
4,23,0


In [9]:
#Transform Amount (Log Scale)

# Handles skewed data (very important in real datasets)

import numpy as np

df['Log_Amount'] = np.log(df['Amount'])
df[['Amount', 'Log_Amount']]

,Amount,Log_Amount
0,500,6.214608
1,20000,9.903488
2,1500,7.313220
3,80000,11.289782
4,300,5.703782


# Encode Categorical Variables (One-Hot)

df = pd.get_dummies(df, columns=['City', 'Device'])
df.head()

#👉 Converts text → numerical format

In [11]:
#Create Behavioral Feature
df['Low_Activity_User'] = df['Past_Transactions'].apply(lambda x: 1 if x < 10 else 0)
df[['Past_Transactions', 'Low_Activity_User']]

#👉 Low activity users = higher fraud risk

,Past_Transactions,Low_Activity_User
0,50,0
1,2,1
2,30,0
3,1,1
4,100,0


In [12]:
#Drop Unnecessary Columns
df_final = df.drop(columns=['Time'])
df_final

#👉 Remove raw features that are no longer needed

,Amount,Past_Transactions,Hour,Is_Night,Log_Amount,City_Delhi,City_Kolkata,City_Mumbai,Device_Desktop,Device_Mobile,Low_Activity_User
0,500,50,10,0,6.214608,False,True,False,False,True,0
1,20000,2,2,1,9.903488,True,False,False,True,False,1
2,1500,30,18,0,7.313220,False,False,True,False,True,0
3,80000,1,1,1,11.289782,True,False,False,False,True,1
4,300,100,23,0,5.703782,False,True,False,True,False,0


In [13]:
df[['Amount', 'Log_Amount']].corr()

,Amount,Log_Amount
Amount,1.000000,0.874508
Log_Amount,0.874508,1.000000


🚀 Full ML Pipeline (Feature Engineering → Train → Predict)
🎯 Goal

Build a fraud detection model using:

Feature engineering
Scaling
Logistic Regression
Prediction with threshold

In [37]:
# 🧪 Step 1: Create Dataset
import pandas as pd

df = pd.DataFrame({
    'Amount': [500, 20000, 1500, 80000, 300, 120000, 700, 40000],
    'Time': ['10:30', '02:15', '18:45', '01:10', '23:50', '03:20', '14:10', '00:30'],
    'City': ['Kolkata', 'Delhi', 'Mumbai', 'Delhi', 'Kolkata', 'Mumbai', 'Delhi', 'Kolkata'],
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Mobile', 'Desktop', 'Mobile', 'Desktop', 'Mobile'],
    'Past_Transactions': [50, 2, 30, 1, 100, 0, 60, 3],
    'Fraud': [0,1,0,1,0,1,0,1]
})

In [38]:
# Step 2: Feature Engineering

import numpy as np

# Extract hour
df['Hour'] = df['Time'].apply(lambda x: int(x.split(':')[0]))

# Night flag
df['Is_Night'] = df['Hour'].apply(lambda x: 1 if x < 6 else 0)

# Log transform
df['Log_Amount'] = np.log(df['Amount'])

# Behavioral feature
df['Low_Activity_User'] = df['Past_Transactions'].apply(lambda x: 1 if x < 10 else 0)

# One-hot encoding
df = pd.get_dummies(df, columns=['City', 'Device'], drop_first=True)

# Drop unused columns
df = df.drop(columns=['Time'])

df.head()

,Amount,Past_Transactions,Fraud,Hour,Is_Night,Log_Amount,Low_Activity_User,City_Kolkata,City_Mumbai,Device_Mobile
0,500,50,0,10,0,6.214608,0,True,False,True
1,20000,2,1,2,1,9.903488,1,False,False,False
2,1500,30,0,18,0,7.313220,0,False,True,True
3,80000,1,1,1,1,11.289782,1,False,False,True
4,300,100,0,23,0,5.703782,0,True,False,False


In [39]:
# ✂️ Step 3: Split Features & Target

X = df.drop(columns=['Fraud'])
y = df['Fraud']

In [40]:
# 🔪 Step 4: Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [41]:
# ⚖️ Step 5: Scaling (IMPORTANT)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = ['Amount', 'Past_Transactions', 'Hour', 'Log_Amount']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])



In [42]:
# 🤖 Step 6: Train Model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [43]:
# 🔮 Step 7: Predict Probabilities

probs = model.predict_proba(X_test)[:, 1]
print("Probabilities:", probs)

Probabilities: [0.65287578 0.94109312]


In [44]:
#🎯 Step 8: Apply Threshold

threshold = 0.5
preds = (probs > threshold).astype(int)

print("Predictions:", preds)
print("Actual:", y_test.values)

Predictions: [1 1]
Actual: [1 1]


In [45]:
# 📊 Step 9: Quick Evaluation

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 1.0


🧠 1. What are Coefficients in Logistic Regression?

In LogisticRegression:

prediction = w1*x1 + w2*x2 + w3*x3 + ... + b

👉 Each w (coefficient) tells:

“How much does this feature influence the prediction?”

In [46]:
#🔍 2. How to Check Coefficients

import pandas as pd

coeff_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
})

coeff_df.sort_values(by='Coefficient', key=abs, ascending=False)

,Feature,Coefficient
4,Log_Amount,0.571153
0,Amount,0.516249
2,Hour,-0.500701
1,Past_Transactions,-0.408463
3,Is_Night,0.295173
5,Low_Activity_User,0.295173
7,City_Mumbai,-0.096162
8,Device_Mobile,0.094446
6,City_Kolkata,0.066299


🧠 Step 1: What This Table Represents

Each row tells:

“If this feature increases, how does fraud probability change?”

Positive coefficient → increases fraud probability
Negative coefficient → decreases fraud probability
Magnitude → strength of impac

🧠 Interpretation (THIS is important)
✅ 1. Amount & Log_Amount (Both High)

👉 Both are strong → expected

Large transactions → more fraud risk ✔
Log version helps model stability ✔

👉 BUT:

These are highly correlated → may be redundant

⚠️ 2. Hour = NEGATIVE (-0.49)

👉 This means:

As hour increases → fraud decreases

So:

Early hours (midnight–morning) → more fraud
Daytime → safer

✔ This matches real-world intuition

✅ 3. Past_Transactions = NEGATIVE

👉 More past transactions → less fraud

✔ Strong business signal:

Old users = safer
New users = risky
🟡 Step 3: Medium Importance Features
Feature	Meaning
Is_Night (+0.29)	Night = more fraud ✔
Low_Activity_User (+0.29)	Low activity = risky ✔

👉 These confirm your feature engineering is working 👍

❌ Step 4: Weak / Low Impact Features
Feature	Coefficient
City_Mumbai	-0.097
Device_Mobile	+0.09
Device_Desktop	-0.09
City_Kolkata	+0.06
City_Delhi	+0.03

👉 These are close to 0

Interpretation:
Location & device → not very useful here
Model is not learning strong patterns from them
⚠️ Step 5: Important Observations (Critical Thinking)
🔥 1. Redundant Features
Amount and Log_Amount both high
👉 Likely correlated

✔ Action:

check correlation → drop one → test performance
🔥 2. Feature Interaction Hidden
Hour (negative) + Is_Night (positive)

👉 Both capturing same idea

✔ Possible redundancy

🔥 3. Weak Features Can Be Removed

👉 Try removing:

City columns
Device columns

And check performance

🧪 Step 6: What YOU should do next
✔ Experiment 1: Remove weak features
drop City + Device → retrain → compare accuracy
✔ Experiment 2: Remove redundancy

Try:

keep only Log_Amount
remove Amount
✔ Experiment 3: Simplify model

Goal:

Fewer features + same performance = better model

🎯 Final Interpretation (Simple Summary)

👉 Your model is saying:

Big transactions → risky
Night transactions → risky
New users → risky
City/device → not important
🧠 One-Line Insight

Your model is correctly learning behavioral risk patterns, not superficial attributes like city or device.

In [47]:
import numpy as np

# Step 1: correlation matrix
corr_matrix = X_train.select_dtypes(include=['number']).corr()

# Step 2: find highly correlated pairs
threshold = 0.9

high_corr = []

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr.append((col1, col2))

print(high_corr)

[('Is_Night', 'Amount'), ('Log_Amount', 'Amount'), ('Log_Amount', 'Is_Night'), ('Low_Activity_User', 'Amount'), ('Low_Activity_User', 'Is_Night'), ('Low_Activity_User', 'Log_Amount')]


🎯 One-Line Summary

Only trust correlations that make mathematical or domain sense — ignore those caused by small data.

🧠 What is Variance Inflation Factor (VIF)?
💡 Simple Definition

VIF measures how much a feature is correlated with other features.

👉 It tells:

“Is this feature redundant because other features already explain it?”

🔍 Why VIF is Needed

Earlier you checked:

Correlation between 2 features ✅

But:

👉 What if a feature is correlated with multiple features together?

In [48]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

# Step 1: Copy data
X_vif = X_train.copy()

# Step 2: Clean data
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)
X_vif = X_vif.dropna()
X_vif = X_vif.astype(float)

# Step 3: Compute VIF safely
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns

vif_values = []
for i in range(X_vif.shape[1]):
    try:
        vif = variance_inflation_factor(X_vif.values, i)
    except:
        vif = np.inf  # handle any numerical issue
    
    vif_values.append(vif)

vif_data["VIF"] = vif_values

# Step 4: Replace inf with large number for readability
vif_data["VIF"] = vif_data["VIF"].replace(np.inf, 9999)

# Step 5: Sort
vif_data = vif_data.sort_values(by="VIF", ascending=False)

print(vif_data)

             Feature     VIF
0             Amount  9999.0
1  Past_Transactions  9999.0
2               Hour  9999.0
3           Is_Night  9999.0
4         Log_Amount  9999.0
5  Low_Activity_User  9999.0
6       City_Kolkata  9999.0
7        City_Mumbai  9999.0
8      Device_Mobile  9999.0


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


# VIF VS CORRELEATION COEFFICIENT

🔍 1️⃣ Correlation Coefficient (Pairwise)
💡 What it measures

Relationship between TWO features

Formula (concept)
corr(X,Y)
Example
corr(Amount, Log_Amount) = 0.95

👉 Meaning:

Strong linear relationship between these two
🔍 2️⃣ Variance Inflation Factor (VIF)
💡 What it measures

How much a feature is explained by ALL OTHER features

Formula
VIF
i
	​

=
1−R
i
2
	​

1
	​


👉 Where:

R
2
 = regression of feature i on all other features
⚡ Key Difference (Most Important)
Aspect	Correlation	VIF
Scope	2 features	1 vs ALL features
Detects	Pairwise relation	Multicollinearity
Output	-1 to 1	1 to ∞
Interpretation	Strength + direction	Redundancy level
🧠 Intuition (Very Important)
🔹 Correlation says:

“Are these two features related?”

🔹 VIF says:

“Can this feature be predicted using other features?”

🧠 VIF Values & Interpretation
📊 Example Output
Feature                VIF
--------------------------------

Log_Amount             1.8

Hour                   2.5

Past_Transactions      3.2

Is_Night               5.8

Low_Activity_User      9.5

Amount                15.2

City_Delhi             1.3

Device_Mobile          1.1
🔍 How to Interpret Each Range
🟢 1️⃣ VIF ≈ 1 (No Correlation)
Device_Mobile → 1.1
City_Delhi → 1.3

👉 Meaning:

Independent feature
No redundancy
✔ Safe to keep
🟡 2️⃣ VIF = 2–5 (Moderate Correlation)
Log_Amount → 1.8
Hour → 2.5
Past_Transactions → 3.2

👉 Meaning:

Some relationship with other features
Still acceptable

✔ Usually keep

🟠 3️⃣ VIF = 5–10 (High Correlation)
Is_Night → 5.8
Low_Activity_User → 9.5

👉 Meaning:

Feature is partially explained by others
Might be redundant

⚠️ Investigate:

Derived feature?
Correlated with another feature?
🔴 4️⃣ VIF > 10 (Very High / Problem)
Amount → 15.2

👉 Meaning:

Strong multicollinearity
Feature is almost redundant

❌ Action:

Remove or replace
🚨 5️⃣ VIF = ∞ (Critical Problem)
Amount → inf
Log_Amount → inf

👉 Meaning:

Perfect multicollinearity
Feature can be exactly predicted

❌ MUST remove feature

🧠 Real Interpretation (Your Fraud Case)
Feature	Reason
Amount (15.2)	Correlated with Log_Amount
Is_Night (5.8)	Derived from Hour
Low_Activity_User (9.5)	Derived from Past_Transactions


# 2. One-Hot Encoding (Advanced Insight)

## Problem
Categorical variables cannot be used directly

## Why One-hot works
Removes false ordering assumption

## Trade-off
High dimensionality → curse of dimensionality


In [49]:
#It converts a categorical feature into multiple independent binary features

In [54]:
df = pd.DataFrame({
    'Age': [22, 45, 25, 35, 52, 23],
    'Salary': [20000, 80000, 25000, 50000, 90000, 27000],
    'City': ['Kolkata', 'Delhi', 'Mumbai', 'Delhi', 'Mumbai', 'Kolkata'],
    'Fraud': [0,1,0,1,1,0]
})

df_encoded = pd.get_dummies(df, columns=['City'], drop_first=True)
df_encoded

,Age,Salary,Fraud,City_Kolkata,City_Mumbai
0,22,20000,0,True,False
1,45,80000,1,False,False
2,25,25000,0,False,True
3,35,50000,1,False,False
4,52,90000,1,False,True
5,23,27000,0,True,False


# label encoding

In [55]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df = pd.DataFrame({
    'Age': [22, 45, 25, 35, 52, 23],
    'Salary': [20000, 80000, 25000, 50000, 90000, 27000],
    'City': ['Kolkata', 'Delhi', 'Mumbai', 'Delhi', 'Mumbai', 'Kolkata'],
    'Fraud': [0,1,0,1,1,0]
})

df['City_Encoded'] = le.fit_transform(df['City'])

df

,Age,Salary,City,Fraud,City_Encoded
0,22,20000,Kolkata,0,1
1,45,80000,Delhi,1,0
2,25,25000,Mumbai,0,2
3,35,50000,Delhi,1,0
4,52,90000,Mumbai,1,2
5,23,27000,Kolkata,0,1


🧠 When to Use Label Encoding
✅ Use when:
Data has natural order

Example: Low → Medium → High

🧠 7. When NOT to Use One-Hot Encoding

Avoid when:

High cardinality features
Tree-based models (like Random Forest, XGBoost can handle categories better sometimes)
Large-scale datasets




# 3. Sigmoid (Deep Insight)

## Behavior
- z → +∞ → output → 1
- z → -∞ → output → 0

## Limitation
Vanishing gradient problem


🧠 Sigmoid Function — Theory (Deep but Clear)

σ(z)=
1+e
−z
1
	​


💡 1. What is Sigmoid?

The sigmoid function converts any real number (−∞ to +∞) into a value between 0 and 1.

👉 That’s why it’s used to represent probability

📈 2. Shape & Behavior (Intuition)
S-shaped curve (also called logistic curve)
Smooth and continuous
Monotonically increasing
Key behavior:
Input (z)	Output
Large negative (−10)	≈ 0
0	0.5
Large positive (+10)	≈ 1
🔍 3. Why Sigmoid is Used in ML
✅ Converts raw model output → probability

Model gives:

z = w1x1 + w2x2 + ... + b

Sigmoid converts it:

Probability = P(class = 1)
🧠 Interpretation
σ(z) = 0.8 → 80% chance of class 1
σ(z) = 0.2 → 20% chance of class 1
⚙️ 4. Mathematical Insight
🔹 Exponential component
e^(-z)
Controls growth/decay
Makes transformation non-linear
🔹 Output is always bounded
0 < σ(z) < 1

👉 Never exactly 0 or 1 (important for training stability)

🔁 5. Derivative (VERY IMPORTANT)

Sigmoid has a beautiful property:

σ
′
(z)=σ(z)(1−σ(z))

👉 Why important?

Used in Gradient Descent
Makes training mathematically convenient
🧠 6. Decision Boundary Role

In classification:

If σ(z) > 0.5 → Class 1  
Else → Class 0

👉 0.5 corresponds to:

z = 0

So:

Decision boundary happens when model output = 0

⚠️ 7. Limitations (Very Important)
❌ 1. Vanishing Gradient Problem

For large values:

z → +∞ → σ(z) ≈ 1  
z → -∞ → σ(z) ≈ 0

👉 Derivative becomes very small
👉 Learning slows down

❌ 2. Not Zero-Centered

Output range:

0 to 1

👉 Always positive
👉 Can slow optimization

🔥 8. Where Sigmoid is Used
Use Case	Why
Logistic Regression	Probability output
Binary classification	Yes/No prediction
Neural networks (output layer)	Probability
🏦 9. Real-World Example (Fraud Detection)

Model computes:

z = 2.0

Sigmoid:

σ(2.0) = 0.88

👉 Interpretation:

88% probability of fraud

In [56]:
import numpy as np

def sigmoid(z):
    return 1/(1+np.exp(-z))

z = np.linspace(-10,10,100)
sigmoid(z)[:5]

array([4.53978687e-05, 5.55606489e-05, 6.79983174e-05, 8.32200197e-05,
       1.01848815e-04])


# 4. Binary Cross Entropy (Deep Insight)

## Why heavy penalty?
Log function explodes for wrong confident predictions

Example:
log(0.1) → large negative → high loss

## Interpretation
Model is punished more when confidently wrong


🧠 Binary Cross Entropy (BCE) — Clear Theory

L=−[ylog(p)+(1−y)log(1−p)]

💡 1. What is Binary Cross Entropy?

Binary Cross Entropy is a loss function used to measure how well a model’s predicted probability matches the actual binary label (0 or 1).

🧠 3. Intuition (Very Important)

👉 BCE answers:

“How wrong is the model’s probability prediction?”

📊 4. Two Key Cases
✅ Case 1: Actual = 1

Loss becomes:

L = -log(p)
p (prediction)	Loss
0.9	Low ✅
0.5	Medium
0.1	High ❌

👉 If model predicts low probability for true class → heavy penalty

✅ Case 2: Actual = 0

Loss becomes:

L = -log(1 - p)
p (prediction)	Loss
0.1	Low ✅
0.5	Medium
0.9	High ❌

🧠 One-Line Summary

Binary Cross Entropy measures how far predicted probabilities are from actual binary outcomes, heavily penalizing confident mistakes.

In [57]:
def bce(y, p):
    return -(y*np.log(p)+(1-y)*np.log(1-p))

print("Good prediction:", bce(1,0.9))
print("Bad prediction:", bce(1,0.1))

Good prediction: 0.10536051565782628
Bad prediction: 2.3025850929940455



# 5. Softmax (Advanced Theory)

## Why sum = 1?
Normalization across all classes

## Key Insight
All classes compete:
Increasing one probability decreases others

## Softmax vs Sigmoid
- Sigmoid → independent
- Softmax → dependent


💡 1. What is Softmax?

Softmax converts a list of raw model scores (logits) into a probability distribution across multiple classes.

🔍 2. Why We Need It

In multi-class problems, the model outputs something like:

[2.0, 1.0, 0.1]

👉 These are not probabilities

Softmax converts them into:

[0.66, 0.24, 0.10]

✔ All values between 0 and 1
✔ Sum = 1

🧠 7. When to Use Softmax

Use when:

Multi-class classification
Only one class is correct

In [59]:
def softmax(z):
    exp = np.exp(z)
    return exp/np.sum(exp)

print(softmax(np.array([2.0,1.0,0.1])))

[0.65900114 0.24243297 0.09856589]



# 6. Decision Boundary

## Insight
Model learns boundary separating classes

Linear vs Non-linear boundaries


💡 1. What is a Decision Boundary?

A decision boundary is the line (or curve) that separates different classes in a dataset.

👉 It answers:

“On which side does this data point belong?”

In [ ]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

X = df_encoded[['Age','Salary']].values
y = df_encoded['Fraud'].values

model = LogisticRegression()
model.fit(X,y)

plt.scatter(X[:,0], X[:,1], c=y)

x_vals = np.linspace(min(X[:,0]), max(X[:,0]),100)
y_vals = -(model.coef_[0][0]*x_vals + model.intercept_) / model.coef_[0][1]

plt.plot(x_vals,y_vals)
plt.title("Decision Boundary")
plt.show()


# 7. Probability Threshold (Business Impact)

## Insight
Threshold changes business decision

Lower threshold → more positives (high recall)
Higher threshold → fewer positives (high precision)


💡 1. What is Probability Threshold?

A probability threshold is the cutoff value used to convert a model’s predicted probability into a final class (0 or 1).

🔍 2. How It Works

Model outputs:

Probability = 0.72

Now apply threshold:

If probability > threshold → Class 1  
Else → Class 0

🎯 8. When to Use What
Use Case	Threshold
Fraud detection	High (0.7–0.9)
Disease detection	Low (0.2–0.4)
Spam detection	Medium (~0.5)

In [79]:
X = df_encoded[['Age','Salary']].values
probs = model.predict_proba(X)[:,1]
print(probs.min(), probs.max())

for t in [0.3,0.5,0.8]:
    preds = (probs > t).astype(int)
    print("Threshold:",t)
    print(preds)

1.610588617319308e-12 1.0
Threshold: 0.3
[0 1 0 1 1 0]
Threshold: 0.5
[0 1 0 1 1 0]
Threshold: 0.8
[0 1 0 1 1 0]



# 🚀 Mini Project: Fraud Detection Pipeline

Steps:
1. Encode data
2. Train model
3. Predict probability
4. Apply threshold


In [76]:
# full pipeline
X = df_encoded[['Age','Salary']].values
y = df_encoded['Fraud'].values

model.fit(X,y)
probs = model.predict_proba(X)[:,1]

threshold = 0.5
preds = (probs > threshold).astype(int)

print("Probabilities:", probs)
print("Predictions:", preds)

Probabilities: [1.61058862e-12 1.00000000e+00 2.48426798e-09 9.99999954e-01
 1.00000000e+00 4.68265257e-08]
Predictions: [0 1 0 1 1 0]
